In [1]:
import sys
import os
import json
import platform
from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 60)
print("🚀 TELEPULSE 360° : ENVIRONMENT & LIBRARIES LOADED")
print("=" * 60)
print(f"🐍 Python Version : {platform.python_version()}")
print(f"📦 Pandas Version : {pd.__version__}")
print(f"📦 NumPy Version  : {np.__version__}")
print(f"📂 Workspace Root : {Path.cwd()}")
print("=" * 60)


🚀 TELEPULSE 360° : ENVIRONMENT & LIBRARIES LOADED
🐍 Python Version : 3.14.5
📦 Pandas Version : 3.1.0
📦 NumPy Version  : 2.5.4
📂 Workspace Root : E:\Venom\telepulse-360


In [2]:
# -------------------------------------------------------------------------
# Step 2: Load & Inspect the 14-Column Telecom Data Schema
# -------------------------------------------------------------------------
schema_path = Path("data/data_schema.json")

# Verify file existence
if not schema_path.exists():
    raise FileNotFoundError(f"Schema file not found at: {schema_path.resolve()}")

with open(schema_path, "r", encoding="utf-8") as f:
    schema_dict = json.load(f)

print("=" * 65)
print(f"📊 PROJECT       : {schema_dict.get('project_name')}")
print(f"🎯 DOMAIN        : {schema_dict.get('domain')}")
print(f"📈 TARGET SCALE  : {schema_dict.get('target_records'):,} Transactions")
print(f"🔢 TOTAL COLUMNS : {len(schema_dict.get('schema', []))} Columns")
print("=" * 65)

# Convert Schema List to a Clean Tabular DataFrame
df_schema = pd.DataFrame(schema_dict["schema"])[["column", "type", "description"]]
df_schema.columns = ["Column Name", "Data Type", "Telecom Domain Description"]

# Display DataFrame as an interactive table
df_schema


📊 PROJECT       : TELEPULSE 360°
🎯 DOMAIN        : Pan-India Telecom Intelligence & ARPU Analytics
📈 TARGET SCALE  : 50,000 Transactions
🔢 TOTAL COLUMNS : 14 Columns


,Column Name,Data Type,Telecom Domain Description
0,transaction_id,string,Unique 12-character alpha-numeric transaction ...
1,customer_id,string,Unique subscriber ID for tracking multiple rec...
2,transaction_date,datetime,"Timestamp of the transaction including date, h..."
3,operator,categorical,Telecom service provider name
4,circle,categorical,TRAI designated Licensed Service Area (Telecom...
5,plan_type,categorical,Type and tier of the mobile subscription pack
6,amount_inr,float,Gross Transaction Value (GMV) in Indian Rupees...
7,validity_days,integer,Validity period of the purchased telecom pack ...
8,data_allowance_gb,float,Total cellular data quota bundled in the plan ...
9,daily_avg_usage_gb,float,Average daily subscriber data consumption calc...


In [3]:
# -------------------------------------------------------------------------
# Step 3: Establish TRAI Market Weights, Benchmark ARPU & 22 Circles
# -------------------------------------------------------------------------

# 1. Real-World Indian Telecom Market Share (TRAI Industry Standards)
MARKET_SHARE = {
    "Reliance Jio": 0.44,   # 44% market share (Volume leader)
    "Bharti Airtel": 0.33,  # 33% market share (ARPU leader)
    "Vodafone Idea": 0.19,  # 19% market share (Retention focus)
    "BSNL": 0.04            # 4% market share (Rural/Public service)
}

# 2. Industry Benchmark ARPU (Average Revenue Per User in INR)
BENCHMARK_ARPU = {
    "Bharti Airtel": 214.0, # High premium postpaid & 4G/5G users
    "Reliance Jio": 185.0,  # Massive 4G/5G volume base
    "Vodafone Idea": 150.0, # Value-seeking subscriber mix
    "BSNL": 95.0            # Legacy 2G/3G & voice-first base
}

# 3. All 22 Indian Telecom Circles (TRAI Licensed Service Areas)
CIRCLES = [
    "Maharashtra & Goa", "Delhi NCR", "Mumbai", "Gujarat", "Karnataka",
    "Tamil Nadu", "Andhra Pradesh & Telangana", "Uttar Pradesh (East)",
    "Uttar Pradesh (West)", "West Bengal & Kolkata", "Rajasthan", "Madhya Pradesh & CG",
    "Punjab & Haryana", "Bihar & Jharkhand", "Kerala", "Assam & North East", "Odisha", "Jammu & Kashmir"
]

TOTAL_TARGET_RECORDS = 50000

# 4. Generate Calibration & Volume Allocation Summary Table
calibration_data = []
for op, share in MARKET_SHARE.items():
    expected_rows = int(TOTAL_TARGET_RECORDS * share)
    target_arpu = BENCHMARK_ARPU[op]
    est_monthly_gmv_lakhs = (expected_rows * target_arpu) / 100000.0

    calibration_data.append({
        "Telecom Operator": op,
        "Market Share": f"{share * 100:.0f}%",
        "Target ARPU (INR)": f"Rs. {target_arpu:.0f}",
        "Allocated 50k Volume": f"{expected_rows:,} rows",
        "Est. Sample GMV": f"Rs. {est_monthly_gmv_lakhs:.1f} Lakhs"
    })

df_calibration = pd.DataFrame(calibration_data)

print("=" * 70)
print(f"🌐 TRAI TELECOM CALIBRATION MATRIX (TOTAL SCALE: {TOTAL_TARGET_RECORDS:,} RECORDS)")
print(f"📍 TOTAL TELECOM CIRCLES REGISTERED : {len(CIRCLES)} Circles")
print("=" * 70)

# Display interactive calibration table
df_calibration


🌐 TRAI TELECOM CALIBRATION MATRIX (TOTAL SCALE: 50,000 RECORDS)
📍 TOTAL TELECOM CIRCLES REGISTERED : 18 Circles


,Telecom Operator,Market Share,Target ARPU (INR),Allocated 50k Volume,Est. Sample GMV
0,Reliance Jio,44%,Rs. 185,"22,000 rows",Rs. 40.7 Lakhs
1,Bharti Airtel,33%,Rs. 214,"16,500 rows",Rs. 35.3 Lakhs
2,Vodafone Idea,19%,Rs. 150,"9,500 rows",Rs. 14.2 Lakhs
3,BSNL,4%,Rs. 95,"2,000 rows",Rs. 1.9 Lakhs


In [4]:
# =========================================================================
# DAY 2 - STEP 1: OPERATOR TARIFF PACKS & PAYMENT GATEWAY ARCHITECTURE
# =========================================================================

# 1. Realistic Tariff Catalog across Jio, Airtel, Vi, and BSNL
OPERATOR_PLANS = {
    "Reliance Jio": [
        {"plan_type": "5G Unlimited Booster", "amount": 61, "validity_days": 28, "data_gb": 10.0},
        {"plan_type": "Daily Data (1.5GB/day)", "amount": 239, "validity_days": 28, "data_gb": 42.0},
        {"plan_type": "Daily Data (2GB/day)", "amount": 299, "validity_days": 28, "data_gb": 56.0},
        {"plan_type": "Daily Data (1.5GB/day)", "amount": 666, "validity_days": 84, "data_gb": 126.0},
        {"plan_type": "Annual Plan (365 Days)", "amount": 2999, "validity_days": 365, "data_gb": 912.5},
        {"plan_type": "Budget Voice / Top-Up", "amount": 155, "validity_days": 28, "data_gb": 2.0}
    ],
    "Bharti Airtel": [
        {"plan_type": "5G Unlimited Booster", "amount": 65, "validity_days": 28, "data_gb": 12.0},
        {"plan_type": "Daily Data (1.5GB/day)", "amount": 265, "validity_days": 28, "data_gb": 42.0},
        {"plan_type": "Daily Data (2GB/day)", "amount": 359, "validity_days": 28, "data_gb": 56.0},
        {"plan_type": "Daily Data (1.5GB/day)", "amount": 719, "validity_days": 84, "data_gb": 126.0},
        {"plan_type": "Annual Plan (365 Days)", "amount": 3359, "validity_days": 365, "data_gb": 912.5},
        {"plan_type": "Budget Voice / Top-Up", "amount": 179, "validity_days": 28, "data_gb": 2.0}
    ],
    "Vodafone Idea": [
        {"plan_type": "Daily Data (1.5GB/day)", "amount": 249, "validity_days": 28, "data_gb": 42.0},
        {"plan_type": "Daily Data (2GB/day)", "amount": 319, "validity_days": 28, "data_gb": 56.0},
        {"plan_type": "Daily Data (1.5GB/day)", "amount": 719, "validity_days": 84, "data_gb": 126.0},
        {"plan_type": "Annual Plan (365 Days)", "amount": 2899, "validity_days": 365, "data_gb": 547.5},
        {"plan_type": "Budget Voice / Top-Up", "amount": 155, "validity_days": 28, "data_gb": 1.0}
    ],
    "BSNL": [
        {"plan_type": "Budget Voice / Top-Up", "amount": 107, "validity_days": 35, "data_gb": 3.0},
        {"plan_type": "Daily Data (2GB/day)", "amount": 199, "validity_days": 30, "data_gb": 60.0},
        {"plan_type": "Daily Data (1.5GB/day)", "amount": 397, "validity_days": 150, "data_gb": 300.0},
        {"plan_type": "Annual Plan (365 Days)", "amount": 1999, "validity_days": 365, "data_gb": 600.0}
    ]
}

# 2. Payment Instrument Usage & Real Failure Rate Benchmark (NPCI / Industry Data)
PAYMENT_INSTRUMENTS = {
    "UPI (PhonePe)": {"share": 0.46, "base_failure_rate": 0.025, "mean_latency_ms": 780},
    "UPI (Google Pay)": {"share": 0.35, "base_failure_rate": 0.028, "mean_latency_ms": 820},
    "UPI (Paytm)": {"share": 0.11, "base_failure_rate": 0.038, "mean_latency_ms": 950},
    "Credit/Debit Card": {"share": 0.05, "base_failure_rate": 0.062, "mean_latency_ms": 1950},
    "Net Banking": {"share": 0.03, "base_failure_rate": 0.075, "mean_latency_ms": 3200}
}

print("=" * 70)
print(f"📦 OPERATORS CONFIGURED      : {len(OPERATOR_PLANS)} (Jio, Airtel, Vi, BSNL)")
print(f"📋 TOTAL TARIFF PACKS        : {sum(len(p) for p in OPERATOR_PLANS.values())} Distinct Plans")
print(f"💳 PAYMENT MODES REGISTERED  : {len(PAYMENT_INSTRUMENTS)} Instruments (UPI, Cards, NetBanking)")
print("=" * 70)

# Display Plans Overview Table
plans_summary = []
for op, plans in OPERATOR_PLANS.items():
    for p in plans:
        plans_summary.append({
            "Operator": op,
            "Plan Category": p["plan_type"],
            "Price (INR)": f"Rs. {p['amount']}",
            "Validity": f"{p['validity_days']} Days",
            "Total Data Quota": f"{p['data_gb']} GB"
        })

df_plans = pd.DataFrame(plans_summary)
df_plans.head(10)


📦 OPERATORS CONFIGURED      : 4 (Jio, Airtel, Vi, BSNL)
📋 TOTAL TARIFF PACKS        : 21 Distinct Plans
💳 PAYMENT MODES REGISTERED  : 5 Instruments (UPI, Cards, NetBanking)


,Operator,Plan Category,Price (INR),Validity,Total Data Quota
0,Reliance Jio,5G Unlimited Booster,Rs. 61,28 Days,10.0 GB
1,Reliance Jio,Daily Data (1.5GB/day),Rs. 239,28 Days,42.0 GB
2,Reliance Jio,Daily Data (2GB/day),Rs. 299,28 Days,56.0 GB
3,Reliance Jio,Daily Data (1.5GB/day),Rs. 666,84 Days,126.0 GB
4,Reliance Jio,Annual Plan (365 Days),Rs. 2999,365 Days,912.5 GB
5,Reliance Jio,Budget Voice / Top-Up,Rs. 155,28 Days,2.0 GB
6,Bharti Airtel,5G Unlimited Booster,Rs. 65,28 Days,12.0 GB
7,Bharti Airtel,Daily Data (1.5GB/day),Rs. 265,28 Days,42.0 GB
8,Bharti Airtel,Daily Data (2GB/day),Rs. 359,28 Days,56.0 GB
9,Bharti Airtel,Daily Data (1.5GB/day),Rs. 719,84 Days,126.0 GB


In [5]:
# =========================================================================
# DAY 2 - STEP 2: SYNTHESIZE 50,000+ RECORDS ACROSS ALL 14 COLUMNS
# =========================================================================
import sys
from pathlib import Path

# Add project root to path for modular import
if str(Path.cwd()) not in sys.path:
    sys.path.append(str(Path.cwd()))

from scripts.data_generator import generate_telecom_dataset

# 1. Execute 50,000 synthetic transaction engine
df_telecom = generate_telecom_dataset(
    num_records=50000,
    num_customers=18000,
    output_path="data/raw/telecom_transactions.csv",
    seed=42
)

# 2. Structural & Statistical Integrity Verification
print("=" * 65)
print(f"📊 DATASET SHAPE : {df_telecom.shape[0]:,} Rows × {df_telecom.shape[1]} Columns")
print("=" * 65)

print("\n🏢 OPERATOR MARKET SHARE VERIFICATION (TRAI Target vs Actual):")
print(df_telecom["operator"].value_counts(normalize=True).mul(100).round(2).astype(str) + "%")

print("\n💳 PAYMENT STATUS DISTRIBUTION:")
print(df_telecom["payment_status"].value_counts(normalize=True).mul(100).round(2).astype(str) + "%")

print("\n📶 NETWORK GENERATION DISTRIBUTION:")
print(df_telecom["network_generation"].value_counts(normalize=True).mul(100).round(2).astype(str) + "%")

# 3. Preview First 5 Records
df_telecom.head()


🚀 Initializing TELEPULSE 360° Data Synthesis Engine (50,000 records)...
✅ Generated 50,000 transactions (7.49 MB)
📁 Output Saved: data/raw/telecom_transactions.csv
📊 DATASET SHAPE : 50,000 Rows × 14 Columns

🏢 OPERATOR MARKET SHARE VERIFICATION (TRAI Target vs Actual):
operator
Reliance Jio     44.08%
Bharti Airtel    33.13%
Vodafone Idea    18.82%
BSNL              3.97%
Name: proportion, dtype: str

💳 PAYMENT STATUS DISTRIBUTION:
payment_status
Success            95.71%
Gateway Timeout     1.49%
User Abandoned      1.46%
Bank Decline        1.34%
Name: proportion, dtype: str

📶 NETWORK GENERATION DISTRIBUTION:
network_generation
4G LTE               52.02%
5G Standalone        25.73%
5G Non-Standalone    17.19%
3G/2G Legacy          5.06%
Name: proportion, dtype: str


,transaction_id,customer_id,transaction_date,operator,circle,plan_type,amount_inr,validity_days,data_allowance_gb,daily_avg_usage_gb,payment_gateway,payment_status,latency_ms,network_generation
0,TXN202600000001,CUST_003649,2026-08-18 19:43:47,Reliance Jio,Uttar Pradesh (West),Daily Data (1.5GB/day),239.0,28,42.0,1.52,UPI (PhonePe),Success,411,5G Standalone
1,TXN202600000002,CUST_000870,2026-09-27 17:51:55,Bharti Airtel,Jammu & Kashmir,Daily Data (1.5GB/day),265.0,28,42.0,1.22,UPI (PhonePe),Success,676,5G Non-Standalone
2,TXN202600000003,CUST_005095,2026-09-18 07:54:22,Reliance Jio,Rajasthan,5G Unlimited Booster,61.0,28,10.0,0.10,UPI (Google Pay),Bank Decline,2266,5G Standalone
3,TXN202600000004,CUST_011851,2026-08-30 19:05:54,Bharti Airtel,Mumbai,5G Unlimited Booster,65.0,28,12.0,0.10,UPI (PhonePe),Success,556,5G Non-Standalone
4,TXN202600000005,CUST_011955,2026-09-04 18:43:41,Reliance Jio,Madhya Pradesh & CG,Daily Data (1.5GB/day),239.0,28,42.0,1.99,UPI (Google Pay),Success,851,5G Standalone
